# 03. Agent Skills — 필요할 때만 읽는 절차

## 학습 목표

- 시스템 프롬프트, 도구(Tool), Skill의 역할과 차이를 명확히 설명할 수 있습니다.
- 다섯 줄짜리 `SKILL.md`를 직접 작성하고, Agent가 그 파일을 실제로 읽는지 확인합니다.
- `description`을 바꿨을 때 활성화 여부가 어떻게 달라지는지, 경쟁하는 Skill 셋이 있는 환경에서 여러 번 실행해 비교합니다.
- 본문에 출력 형식을 지정하고, 긴 지침을 참조 파일로 분리하며, 단어는 겹치지만 읽으면 안 되는 질문(near-miss)을 시험합니다.
- 내 업무에 쓸 수 있는 나만의 Skill을 하나 만들고 검증합니다.

## 호출 규모

이 노트북은 **실제 모델**만 호출합니다. `Day02_new/.env` 파일에 `ANTHROPIC_API_KEY`가 설정되어 있지 않으면 첫 번째 코드 셀에서 안내 메시지와 함께 실행이 멈춥니다. 유료 호출을 하는 셀은 실행 직전에 예상 호출 규모를 한 줄로 알려 드립니다.

기본값인 `RUN_OPTIONAL = False` 상태에서 실행하면, 각 유료 셀이 실행 직전에 알려 주는 예상 호출 수(`announce`의 `model_calls`)를 모두 더해 **약 132회**로 추정됩니다. 이 숫자는 셀마다 적어 둔 예상치의 합계이므로 추정값이며, Agent가 도구를 몇 번 부르느냐에 따라 실제 호출 수는 조금씩 달라집니다. 이 중 경쟁 Skill 셋 비교 셀 하나가 약 70회로 가장 큽니다. 시간이 오래 걸리는 대규모 측정(질문 20개 × 3회, 이전에 저자가 한 번 실행했을 때의 기록은 81회)과 Skill 있음/없음 비교(질문 3개 × 두 조건)는 `RUN_OPTIONAL = True`로 직접 바꿔야만 실행되며, 이 두 셀의 예상 호출 수를 더하면 약 100회입니다. 모든 API 비용은 본인의 키로 청구됩니다.

위의 약 132회와 약 100회는 셀이 알려 주는 **예상치**이고, 아래는 저자가 실제로 실행해 남긴 **측정 기록**입니다. 이 기록은 모두 한 대의 컴퓨터에서 한 번 실행한 값이라, 여러분의 실행에서는 호출 수와 시간이 달라질 수 있습니다.

- **2026-11-02 실행 기록, 기본 경로(`RUN_OPTIONAL = False`)**: 한 번 실행에서 모델 호출 87회, 전체 126.3초였습니다. 예상치 약 132회보다 적게 나왔는데, 짧은 관찰 셀마다 Agent가 도구를 부르는 횟수가 예상보다 적었기 때문입니다.
- **2026-11-02 실행 기록, 선택 측정 두 셀(`RUN_OPTIONAL = True`)**: 한 번 실행에서 전체 트리거 측정 셀이 모델 호출 79회, 253.2초(약 4분 13초)였고, Skill 있음/없음 비교 셀이 17회, 46.0초였습니다. 두 셀을 합치면 96회로, 예상치 약 100회와 비슷합니다. 같은 실행에서 선택 측정 두 셀을 뺀 본문은 88회였습니다. 기본 경로의 87회와 한 번 차이 나는 것은 Agent가 도구를 부르는 횟수가 실행마다 조금씩 달라지기 때문으로 보이며, 정확한 원인은 확인하지 않았습니다.
- 앞 문단에서 말한 이전 기록 81회와 18회는 코드를 고치기 전의 별도 실행에서 나온 값이고, 위 2026-11-02 실행 기록의 79회와 17회와는 다른 실행입니다.

| 구분 | 셀 수 | 모델 호출(예상) |
|---|---:|---|
| 무료 (설정, 도우미 정의, 파일 쓰기, 폴더 규칙 검사 등) | 11 | 0 |
| 짧은 관찰 (Agent 1회 실행, 셀마다 약 3회) | 8 | 약 24 |
| 짧은 관찰 (질문 2개 또는 Skill 있음/없음 비교, 셀마다 약 6회) | 3 | 약 18 |
| 경쟁 Skill 셋 비교 (질문 4개 × 설명 2가지 × 3회) | 1 | 약 70 |
| 소규모 트리거 테스트 (4문항) | 1 | 약 12 |
| 제공 Skill로 RAG 실행 및 검증 오류 재시도 | 2 | 약 3 + 약 5 |
| **기본 경로 합계** | 15 | **약 132** |
| 선택 측정 두 개 (`RUN_OPTIONAL`) | 2 | 약 80 + 약 20 = 약 100 |
| 2026-11-02 실행 기록: 기본 경로, 1회 실행 | 15 | 87회, 126.3초 |
| 2026-11-02 실행 기록: 전체 트리거 측정 셀, 1회 실행 | 1 | 79회, 253.2초 |
| 2026-11-02 실행 기록: Skill 있음/없음 비교 셀, 1회 실행 | 1 | 17회, 46.0초 |

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 왜 Skill인가: 프롬프트, 도구, Skill | 5 |
| 2 | 다섯 줄 Skill을 손으로 쓰기 | 5 |
| 3 | `SKILL.md` 규칙과 점진적 공개 | 6 |
| 4 | 설명을 바꿔 활성화 비교, 경쟁 Skill 셋 | 12 |
| 5 | 출력 형식 더하기, 참조 파일 나누기 | 7 |
| 6 | 트리거 테스트와 near-miss | 5 |
| 7 | 읽으면 안 되는 질문 | 6 |
| 8 | 출력 평가, Skill이 맡지 않는 것 | 2 |
| 9 | 직접 해보기: 내 업무 Skill | 9 |
| 10 | 정리와 확인 질문 | 3 |

각 코드 셀 첫 줄에 적힌 표지는 다음과 같은 뜻입니다.
- `# [따라하기]`: 셀을 그대로 실행하고 결과를 관찰합니다.
- `# [바꿔보기]`: 바꿀 값을 확인하고 **결과를 먼저 예측한 뒤** 실행하여 예측과 비교합니다.
- `# [깨뜨리기]`: 일부러 규칙을 어기거나 실패 상황을 만들어 오류가 어떻게 처리되는지 확인합니다.
- `# [내 것 만들기]`: 예시 값을 내 업무 주제로 바꾸어 직접 실습합니다.

이 노트북에서는 손으로 조작한 가짜 메시지를 쓰지 않고 실제 모델 응답만 관찰합니다.

## 1절. 왜 Skill인가

Agent에게 "회의록은 이렇게 요약하고, 표는 저렇게 계산하라"는 업무 규칙을 모두 가르친다고 생각해 봅시다. 가장 단순한 방법은 모든 규칙을 **시스템 프롬프트**에 적어 넣는 것입니다. 하지만 이렇게 하면 질문이 한 줄짜리 단순 사실 확인일 때조차 모든 업무 규칙이 매번 모델 호출에 실려 갑니다. 토큰 비용이 늘어날 뿐 아니라, 서로 다른 업무 규칙끼리 부딪히기도 쉽습니다.

Skill은 이 문제를 **필요할 때만 꺼내 읽는 절차 문서**로 해결합니다. 모델은 평소에는 각 Skill의 이름과 짧은 설명(메타데이터)만 보고 있다가, 사용자의 질문이 그 설명과 맞을 때만 파일 본문을 열어 읽습니다.

| 구분 | 시스템 프롬프트 | 도구 (Tool) | Skill |
|---|---|---|---|
| **무엇인가** | 항상 읽히는 기본 지침 | 호출하면 코드가 실행되는 함수 | 필요할 때만 열어 읽는 절차 문서 |
| **모델에 전달되는 시점** | 모든 모델 호출마다 전체 내용 | 이름과 인자 스키마는 항상 전달 | 이름과 설명은 항상 전달, 본문은 읽을 때만 전달 |
| **강제력** | 없음 (모델이 따를지 결정) | 실행 자체는 코드가 보장, 호출 여부는 모델이 결정 | 없음 (읽을지, 따를지 모두 모델이 결정) |
| **따르지 않을 때** | 조용히 무시됨 | 실행 오류가 반환됨 | 아무 오류 없이 넘어감 |

비유하자면 시스템 프롬프트는 벽에 붙여 놓은 상시 규칙표이고, 도구는 손에 쥐여 준 공구함이며, Skill은 서랍 속에 넣어 둔 업무 매뉴얼입니다. 매뉴얼은 표지(이름과 설명)만 보이게 꽂아 두고 필요할 때만 꺼내 읽습니다. 만약 표지 설명이 부실하면 아무도 그 매뉴얼을 꺼내 보지 않을 것입니다.

표의 마지막 줄이 이 노트북 전체를 관통하는 핵심 문제의식입니다. **Skill은 지침일 뿐 강제가 아닙니다.** Skill 파일을 "만들어 두었다"는 것과 모델이 그 파일을 "실제로 읽었다"는 것은 전혀 다른 일입니다. 따라서 모델이 Skill을 끝까지 읽었는지는 반드시 코드로 추적해 확인해야 합니다.

## 2절. 다섯 줄 Skill을 손으로 쓰기

먼저 실습 환경을 준비하는 설정 셀을 실행해 보겠습니다. 수업 중에 우리가 새로 만드는 Skill은 모두 `outputs/skill_sandbox/` 아래에 저장하고, 저장소에 기본 제공되는 `skills/` 폴더는 건드리지 않습니다. 노트북을 처음부터 다시 실행해도 항상 깨끗한 상태에서 시작할 수 있도록 실행할 때마다 샌드박스 폴더를 비우고 새로 만듭니다.

In [ ]:
# [1단계] 필요한 라이브러리와 이 수업의 도우미 모듈을 불러옵니다
import json
import shutil
from datetime import date

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from deepagents.middleware.filesystem import FilesystemPermission
from deepagents.middleware.skills import _list_skills
from langchain_core.callbacks import BaseCallbackHandler

from rag_lab.agent import TrackedSkillsBackend, ask, check_skill_folder
from rag_lab.answer import skill_spec
from rag_lab.corpus import Scope, load_corpus
from rag_lab.retrieval import Retriever
from rag_lab.settings import ROOT, announce, key_status, make_model, require_key

# [2단계] 선택 측정 스위치
# 대규모 측정(질문 20개 x 3회 등)은 True로 바꿔야만 실행됩니다.
# False는 "이번 수업 시간에는 돌리지 않는다"는 의도적인 선택입니다.
RUN_OPTIONAL = False

# [3단계] 기본 Skill 폴더와 실습용 샌드박스 폴더 경로를 정합니다
SKILLS = ROOT / "skills"                      # 기본 제공되는 Skill 폴더 (읽기 전용으로 사용합니다)
SANDBOX = ROOT / "outputs" / "skill_sandbox"  # 실습 중 만드는 Skill을 저장할 샌드박스 폴더입니다

# [4단계] 샌드박스를 비우고 새로 만듭니다 (몇 번을 다시 실행해도 같은 상태에서 시작하기 위해서입니다)
if SANDBOX.exists():
    shutil.rmtree(SANDBOX)
SANDBOX.mkdir(parents=True)

# [5단계] API 키를 확인합니다
require_key()                                 # API 키가 없으면 안내 메시지와 함께 여기서 멈춥니다
print(key_status())

Skill의 실체는 아주 단순합니다. **폴더 하나와 그 안에 들어 있는 `SKILL.md` 파일 하나**가 전부입니다. 파일 맨 위의 `---`로 둘러싸인 부분이 메타데이터(이름과 설명)이고, 그 아래가 실제 절차 본문입니다. 아래 셀을 그대로 실행해 다섯 줄짜리 Skill을 직접 만들어 보세요. 모델 호출은 0회입니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회 — 다섯 줄짜리 Skill 파일을 작성합니다
# [1단계] Skill 폴더를 만듭니다. 폴더 이름(meeting-notes)은 SKILL.md의 name과 같아야 합니다
folder = SANDBOX / "first" / "meeting-notes"
folder.mkdir(parents=True)

# [2단계] SKILL.md 내용을 만듭니다. --- 사이가 메타데이터(name, description), 그 아래가 본문입니다
skill_text = """---
name: meeting-notes
description: 회의록을 결정, 할 일, 미정 세 줄로 요약한다. 회의록 요약을 요청받을 때 쓴다.
---
결정, 할 일, 미정 사항을 각각 한 줄로 쓴다.
"""

# [3단계] 파일로 저장하고, 저장된 내용을 그대로 읽어 출력해 확인합니다
skill_file = folder / "SKILL.md"
skill_file.write_text(skill_text, encoding="utf-8")
print(skill_file.read_text(encoding="utf-8"))

`SKILL.md` 파일이 생성되었습니다. 이제 이 Skill 폴더를 `create_deep_agent`에 연결해 실행하기 전에, (1) 첫 번째 모델 호출에 전달된 시스템 프롬프트 원문을 기록하는 콜백(`FirstPrompt`)과 (2) 에이전트를 한 턴 실행하고 읽은 Skill 목록과 도구 호출 기록을 돌려주는 도우미 함수(`skill_turn`)를 먼저 정의합니다.

In [ ]:
# [1단계] 첫 번째 모델 호출에 전달된 시스템 프롬프트 원문을 기록하는 콜백 정의
class FirstPrompt(BaseCallbackHandler):
    """모델이 첫 번째 호출에서 받은 시스템 프롬프트 원문을 기록하는 콜백입니다.

    Skill의 이름과 설명은 시스템 프롬프트에 처음부터 실려 있고 본문은 실려 있지 않다는 점을
    눈으로 확인하려고 사용합니다. 두 번째 호출부터는 기록하지 않고 첫 호출만 남깁니다.
    """

    def __init__(self):
        self.system = ""

    def on_chat_model_start(self, serialized, messages, **kwargs):
        # 이미 기록했다면(두 번째 호출 이후) 아무것도 하지 않습니다
        if self.system:
            return

        # messages[0][0]이 첫 호출의 첫 메시지(시스템 프롬프트)입니다
        content = messages[0][0].content
        if isinstance(content, str):
            self.system = content
            return

        # 내용이 블록 목록이면 각 블록의 text를 이어 붙여 하나의 문자열로 만듭니다
        texts = []
        for block in content:
            texts.append(block.get("text", ""))
        self.system = "".join(texts)


# [2단계] 지정한 폴더의 Skill을 연결해 Agent를 한 턴 실행하고 기록을 돌려주는 도우미 함수
def skill_turn(root, question):
    """root 폴더를 Skill 위치로 사용하는 Agent에게 질문 하나를 보내고 실행 기록을 돌려줍니다.

    돌려주는 딕셔너리의 키는 다음과 같습니다.
    - answer: 최종 답변 텍스트
    - calls: 모델이 요청한 도구 호출 (이름, 인자) 목록
    - seen: 끝까지 읽힌(활성화된) Skill 이름 목록
    - prompt: 첫 모델 호출의 시스템 프롬프트 원문
    """
    seen = []
    prompt = FirstPrompt()

    # 2-1) TrackedSkillsBackend로 모델이 SKILL.md를 끝까지 읽었는지(seen) 추적하고 파일 쓰기는 차단합니다
    agent = create_deep_agent(
        model=make_model(),
        backend=TrackedSkillsBackend(root, seen.append),
        skills=["/"],
        permissions=[FilesystemPermission(operations=["write"], paths=["/**"], mode="deny")],
    )

    # 2-2) 질문을 넣어 Agent를 실행합니다
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"callbacks": [prompt], "recursion_limit": 30},
    )

    # 2-3) 모델이 요청한 모든 도구 호출(이름, 인자)을 순서대로 모읍니다
    calls = []
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", []):
            calls.append((call["name"], call["args"]))

    # 2-4) 최종 답변은 마지막 메시지의 텍스트입니다
    final_answer = result["messages"][-1].text
    return {
        "answer": final_answer,
        "calls": calls,
        "seen": seen,
        "prompt": prompt.system,
    }


# [3단계] 실행 기록에서 read_file 도구로 읽은 파일 경로만 추려내는 도우미 함수와 실습용 회의록 준비
def read_paths(turn):
    """실행 결과에서 모델이 read_file 도구로 열어 본 파일 경로만 추려냅니다."""
    paths = []
    for name, args in turn["calls"]:
        if name == "read_file":
            paths.append(args.get("file_path"))
    return paths


MEETING = (
    "4월 7일 주간 회의. 신규 고객 온보딩 일정은 5월 12일로 확정했다. "
    "김서연이 4월 11일까지 체크리스트 초안을 만든다. "
    "가격 개편안은 법무 검토 결과를 보고 다시 논의하기로 했다."
)
print("준비 완료. 회의록 예시:", MEETING)

도우미 함수와 회의록 예시(`MEETING`)가 준비되었으니, 이제 방금 만든 다섯 줄짜리 `meeting-notes` Skill을 Agent에게 주고 회의록 요약을 요청해 보겠습니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 3회 — 내가 쓴 Skill을 Agent가 실제로 읽는지 확인합니다
# [1단계] 실행 직전에 규모를 알리고, 다섯 줄짜리 Skill 폴더로 Agent를 한 턴 실행합니다
announce("다섯 줄짜리 Skill을 Agent에게 주고 회의록 요약을 요청합니다", model_calls=3)
question = "다음 회의록을 요약해 주세요.\n" + MEETING
turn = skill_turn(SANDBOX / "first", question)

# [2단계] 모델이 실제로 무엇을 읽었는지 확인합니다
print("읽은 파일 경로:", read_paths(turn))
print("활성화된 Skill(끝까지 읽은 Skill):", turn["seen"])

# [3단계] 이름은 시스템 프롬프트에 있지만 본문 문장은 없다는 점을 확인합니다
name_in_prompt = "meeting-notes" in turn["prompt"]
body_in_prompt = "각각 한 줄로 쓴다" in turn["prompt"]
print("Skill 이름이 초기 시스템 프롬프트에 있는가:", name_in_prompt)
print("Skill 본문 문장이 초기 시스템 프롬프트에 있는가:", body_in_prompt)

# [4단계] 최종 답변을 출력합니다
print("\n[Agent 답변]")
print(turn["answer"])

위 실행 결과에서 다음 세 가지를 함께 확인해 보세요.
- 모델이 `read_file` 도구를 불러 `/meeting-notes/SKILL.md`를 직접 열었나요?
- 답변이 지침대로 세 줄(결정, 할 일, 미정)로 나왔나요?
- **Skill 이름은 처음부터 시스템 프롬프트에 들어 있지만(`True`), 본문 문장은 시스템 프롬프트에 없다(`False`)**는 점을 확인하셨나요?

정말로 초기 시스템 프롬프트(`turn["prompt"]`) 안에 무엇이 어떻게 적혀 있었는지, `"meeting-notes"`가 등장하는 앞뒤 구간을 직접 잘라 눈으로 확인해 봅시다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회 — 초기 시스템 프롬프트에 적재된 Skill 메타데이터 원문 확인하기
# [1단계] 앞 셀에서 기록한 시스템 프롬프트에서 Skill 이름이 처음 나오는 위치를 찾습니다 (없으면 -1)
prompt_text = turn["prompt"]
pos = prompt_text.find("meeting-notes")

# [2단계] 그 위치의 앞뒤 구간을 잘라 Skill 목록이 어떻게 적혀 있는지 눈으로 확인합니다
print("=== 초기 시스템 프롬프트 중 Skill 목록 구간 ===")
if pos != -1:
    start = max(0, pos - 120)
    end = pos + 260
    snippet = prompt_text[start:end]
    print(snippet)
print("==============================================")

# [3단계] 본문 문장은 시스템 프롬프트에 없다는 점을 다시 확인합니다
print("본문 문장('각각 한 줄로 쓴다') 포함 여부:", "각각 한 줄로 쓴다" in prompt_text)

**관찰할 포인트:**
- 초기 시스템 프롬프트에는 `meeting-notes`라는 **이름**과 `회의록을 결정, 할 일, 미정 세 줄로 요약한다...`라는 **`description`(설명)**만 적혀 있고, 본문 내용은 들어 있지 않습니다. 모델은 이 한 줄 설명을 읽고 나서야 `read_file` 도구로 `/meeting-notes/SKILL.md` 본문을 열어 본 것입니다. 이것이 바로 **점진적 공개(Progressive Disclosure)**의 핵심 동작입니다.
- 단, 방금 실행한 에이전트 응답은 **1회 관찰**일 뿐 확률이나 비율이 아닙니다. 같은 셀을 다시 실행하면 모델이 읽는 경로나 답변 말투가 조금씩 달라질 수 있습니다.

## 3절. `SKILL.md` 규칙과 점진적 공개

Skill을 만들 때는 다음 규칙을 지켜야 합니다.

- **`name`**: 영문 소문자, 숫자, 하이픈(`-`)만 사용할 수 있으며 64자 이하여야 합니다. 무엇보다 **폴더 이름과 정확히 일치**해야 합니다.
- **`description`**: 1~1024자 이내로 작성합니다. 모델은 오직 이 설명만 보고 파일을 열어 볼지 결정하므로, **"무엇을 하는가 + 언제 쓰는가 + 이웃 Skill과의 경계(언제 쓰지 않는가)"**를 명확히 담아야 합니다.

**점진적 공개(Progressive Disclosure)의 세 단계**는 다음과 같습니다.
1. **메타데이터(`name`, `description`)**: 시스템 프롬프트에 항상 포함됩니다.
2. **본문(`SKILL.md`)**: 모델이 설명을 보고 필요하다고 판단해 `read_file`을 호출했을 때만 문맥에 들어옵니다.
3. **참조 파일(`references/` 등)**: 본문 안에서 특정 참조 파일을 읽으라고 안내할 때만 추가로 읽어 들입니다.

**활성화(Activation)의 정의:** 이 수업에서는 모델이 `SKILL.md` 파일을 **처음부터 끝까지** 읽었을 때만 해당 Skill이 '활성화되었다'고 정의합니다. 앞부분 몇 줄만 읽다 만 경우는 활성화로 인정하지 않습니다. 앞서 정의한 `skill_turn`의 `seen`(`TrackedSkillsBackend`)이 바로 이 기준으로 활성화를 기록합니다.

예를 들어 규칙을 잘 지킨 메타데이터는 다음과 같은 형태입니다.

```yaml
---
name: meeting-notes          # 폴더 이름과 정확히 일치해야 합니다
description: 회의록을 ...    # 무엇을 + 언제 + (필요하면) 쓰지 않는 때
---
```

비용 관점에서 보면, 메타데이터는 Skill이 열 개든 백 개든 매번 모든 호출에 실려 갑니다. 따라서 메타데이터의 `description`은 핵심만 짧고 정확하게 쓰고, 길거나 특정 상황에서만 필요한 세부 규칙은 본문이나 참조 파일로 분리하는 것이 좋습니다.

만약 작성 규칙을 어기면 어떻게 될까요? 아래 `[깨뜨리기]` 셀에서 폴더 이름과 다른 `name`, 대문자와 밑줄을 섞어 쓴 이름, 비어 있는 `description` 세 가지 잘못된 Skill을 일부러 만들어 확인해 보겠습니다.

In [ ]:
# [깨뜨리기] 무료, 모델 호출 0회 — 규칙을 어긴 Skill 세 가지를 만들어 검사합니다
# [1단계] 일부러 규칙을 어긴 세 가지 경우를 정의합니다: {폴더 이름: (SKILL.md의 name, description)}
BROKEN = SANDBOX / "broken"
cases = {
    "name-mismatch": ("other-name", "이름이 폴더와 다른 Skill."),
    "Bad_Name": ("Bad_Name", "대문자와 밑줄을 쓴 이름."),
    "no-description": ("no-description", ""),
}

# [2단계] 경우마다 폴더와 SKILL.md를 만듭니다
for folder_name, (name, description) in cases.items():
    path = BROKEN / folder_name
    path.mkdir(parents=True)

    # description은 JSON 문자열로 감싸 YAML 파싱 오류가 나지 않게 합니다
    description_text = json.dumps(description, ensure_ascii=False)
    content = (
        f"---\n"
        f"name: {name}\n"
        f"description: {description_text}\n"
        f"---\n"
        f"본문 한 줄.\n"
    )
    (path / "SKILL.md").write_text(content, encoding="utf-8")

# [3단계] Deep Agents가 어떤 Skill을 적재하는지 확인합니다 (규칙 위반이어도 적재되는 경우가 있습니다)
backend = FilesystemBackend(root_dir=BROKEN, virtual_mode=True)
loaded_names = []
for skill in _list_skills(backend, "/"):
    loaded_names.append(skill["name"])
print("Deep Agents가 적재한 목록:", loaded_names)

# [4단계] 우리가 만든 검사기는 세 폴더를 모두 규칙 위반으로 잡아내는지 확인합니다
for folder in sorted(BROKEN.iterdir()):
    problems = check_skill_folder(folder)
    print(f"{folder.name:16s} -> 검사 결과: {problems}")

위 셀을 실행할 때 출력되는 영문 경고는 Deep Agents 패키지가 내보내는 것입니다. 우리가 만든 검사기(`check_skill_folder`)는 세 폴더 모두 규칙 위반으로 정확히 잡아냅니다.

그런데 Deep Agents가 적재한 목록(`_list_skills`)을 자세히 보면, 이름 규칙을 어긴 Skill(`name-mismatch`, `Bad_Name`)도 목록에 그대로 올라와 있는 것을 볼 수 있습니다. Deep Agents는 이름 규칙 위반에 대해 경고만 출력하고 일단 적재하기 때문입니다(`description`이 빈 경우만 건너뜁니다). 이처럼 예외가 발생하지 않고 넘어가는 **조용한 실패**가 있을 수 있으므로, 새 Skill을 만들면 반드시 `check_skill_folder` 검사기를 먼저 돌려 확인해야 합니다.

## 4절. 설명을 바꿔 활성화 비교

모델은 Skill의 `description`만 보고 파일을 열어 볼지 결정합니다. 만약 설명을 `회의록`이라는 단어 하나로 줄여 버리면 어떻게 될까요? 두 가지 질문으로 비교해 보겠습니다.
- **직접 요청**: "다음 회의록을 요약해 주세요."
- **돌려 말한 요청**: "오늘 미팅 내용 정리해 줘."

각 질문을 한 번씩만 실행하므로 이 결과는 **통계적 비율이 아니라 1회 관찰**입니다.

`[바꿔보기]` 셀을 실습할 때는 항상 같은 순서를 따릅니다. 바꿀 값을 확인하고, **실행하기 전에 결과를 먼저 예측해 적은 다음**, 셀을 실행해 내 예측과 실제 결과를 비교해 보세요. 예측이 빗나갔을 때 가장 많이 배울 수 있습니다. 예측을 적지 않고 그냥 실행하면 어떤 결과가 나와도 "원래 그럴 줄 알았다"고 넘어가기 쉽습니다.

먼저 아래 셀을 실행해 Skill 파일을 손쉽게 만들어 주는 `write_skill` 함수와 활성화 확인 함수 `activated`, 그리고 비교할 두 질문을 준비합니다.

In [ ]:
# [1단계] Skill 폴더와 SKILL.md를 손쉽게 만들어 주는 도우미 함수
def write_skill(root, name, description, body, files=None):
    """root/name/SKILL.md 파일을 작성합니다. files는 {상대경로: 내용} 딕셔너리입니다.

    설명(description)은 JSON 문자열로 감싸서 YAML 파싱 오류가 나지 않게 안전하게 기록합니다.
    files에 참조 파일을 넣으면 Skill 폴더 아래에 함께 만들어 줍니다. 만든 Skill 폴더 경로를 돌려줍니다.
    """
    # 1-1) Skill 폴더를 만듭니다 (이미 있어도 오류 없이 넘어갑니다)
    folder = root / name
    folder.mkdir(parents=True, exist_ok=True)

    # 1-2) SKILL.md를 만듭니다: 메타데이터(name, description) + 본문
    description_text = json.dumps(description, ensure_ascii=False)
    skill_md = (
        f"---\n"
        f"name: {name}\n"
        f"description: {description_text}\n"
        f"---\n"
        f"{body}\n"
    )
    (folder / "SKILL.md").write_text(skill_md, encoding="utf-8")

    # 1-3) 참조 파일이 있으면 하위 폴더까지 만들면서 함께 저장합니다
    extra_files = files or {}
    for relative, content in extra_files.items():
        target = folder / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
    return folder


# [2단계] 활성화 확인 도우미 함수
def activated(turn, name="meeting-notes"):
    """해당 턴에서 지정한 Skill이 끝까지 읽혔는지(활성화되었는지) 확인합니다."""
    return name in turn["seen"]


# [3단계] 이후 셀에서 함께 쓸 Skill 본문과 비교할 두 질문 준비
BODY = "결정, 할 일, 미정 사항을 각각 한 줄로 쓴다."
QUESTIONS = {
    "직접": "다음 회의록을 요약해 주세요.\n" + MEETING,
    "돌려 말함": "오늘 미팅 내용 정리해 줘.\n" + MEETING,
}
print("질문 두 개 준비 완료:", list(QUESTIONS.keys()))

먼저 `description`에 `"회의록"`이라는 한 단어만 적어 두었을 때, 직접 요청과 돌려 말한 요청에서 각각 Skill이 활성화되는지 실행해 보겠습니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 6회 — 설명이 '회의록' 한 단어뿐일 때 두 질문을 실행합니다
# [1단계] 설명이 '회의록' 한 단어뿐인 Skill을 만듭니다
announce("설명 '회의록' 한 단어로 질문 두 개를 실행합니다", model_calls=6)
write_skill(SANDBOX / "desc-weak", "meeting-notes", "회의록", BODY)

# [2단계] 직접 요청과 돌려 말한 요청을 각각 한 번씩 실행해 기록을 모읍니다
weak = {}
for label, question in QUESTIONS.items():
    weak[label] = skill_turn(SANDBOX / "desc-weak", question)

# [3단계] 질문마다 활성화 여부와 읽은 경로를 출력합니다
for label, weak_turn in weak.items():
    is_active = activated(weak_turn)
    paths = read_paths(weak_turn)
    print(f"{label:6s} | 활성화={is_active} | 읽은 경로={paths}")

한 단어 설명의 결과를 확인했으니, 이번에는 `description`을 **"무엇을 하는가 + 언제 쓰는가"**가 모두 드러나도록 구체적으로 바꾸어 같은 두 질문을 다시 실행해 비교해 보겠습니다. 실행하기 전에 `PREDICTION`에 여러분의 예측을 먼저 적어 보세요.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 6회 — description을 "무엇을 + 언제"로 구체화하여 비교합니다
# 바꿀 값: STRONG_DESCRIPTION. 먼저 예측을 적고 실행하세요: 어느 질문의 활성화 여부가 바뀔까요?
# [1단계] 내 예측과 구체적인 설명(무엇을 + 언제)을 정합니다
PREDICTION = "여기에 예측을 적습니다"
STRONG_DESCRIPTION = (
    "회의록이나 미팅 내용을 결정, 할 일, 미정 세 줄로 요약한다. "
    "회의록 요약이나 미팅 정리를 요청받을 때 쓴다."
)

# [2단계] 구체적인 설명의 Skill을 만들고 같은 두 질문을 실행합니다
announce("설명을 구체적으로 바꾸고 같은 질문 두 개를 실행합니다", model_calls=6)
write_skill(SANDBOX / "desc-strong", "meeting-notes", STRONG_DESCRIPTION, BODY)

strong = {}
for label, question in QUESTIONS.items():
    strong[label] = skill_turn(SANDBOX / "desc-strong", question)

# [3단계] 앞 셀(weak)과 이번 셀(strong)의 활성화 여부를 한 표로 비교합니다
print("내 예측:", PREDICTION)
print(f"{'질문':8s} {'설명=회의록':>12s} {'설명=무엇을+언제':>16s}")
for label in QUESTIONS:
    weak_act = str(activated(weak[label]))
    strong_act = str(activated(strong[label]))
    print(f"{label:8s} {weak_act:>12s} {strong_act:>16s}")

`description`은 Skill의 얼굴입니다. 본문 절차가 아무리 훌륭해도 설명이 부실하면 모델이 파일을 꺼내 읽지 않습니다.

위 표에서 두 열의 결과가 달라진 질문이 있나요? 만약 두 열이 모두 `True`로 같다면, 모델이 `회의록`이라는 한 단어 설명과 `meeting-notes`라는 이름만 보고도 충분히 역할을 짐작해 읽어 본 것입니다. 이 표는 질문당 한 번씩만 실행한 관찰이므로, 이것만으로 "설명이 길수록 항상 더 잘 활성화된다"고 단정할 수는 없습니다. 활성화 비율을 말하려면 같은 질문을 여러 번 반복해 측정해야 합니다.

또한 위 실험에서 차이가 잘 드러나지 않았다면 더 중요한 구조적 이유가 하나 있습니다. 폴더 안에 **Skill이 단 하나뿐**이면, 모델 입장에서는 설명이 조금 부실하더라도 고를 수 있는 후보가 그것밖에 없습니다. 설명(`description`)의 진짜 위력은 **비슷한 주제를 다루는 경쟁 Skill들이 함께 있을 때** 드러납니다. 바로 아래에서 단어가 겹치는 Skill 세 개를 한 폴더에 넣고 비교해 보겠습니다.

### 경쟁하는 Skill 셋

이번에는 회의와 관련된 세 가지 Skill(`meeting-notes` 회의록 요약, `meeting-schedule` 회의 일정 조율, `meeting-followup` 회의 후 안내 메일 작성)을 한 폴더에 함께 넣어 두겠습니다. 그리고 설명을 두 가지 버전으로 만들어 비교합니다.
- **막연한 설명(`막연`)**: 세 Skill 모두 "회의 내용/일정/후속 작업을 다룬다"처럼 경계 없이 두루뭉술하게 적습니다.
- **구체적인 설명(`구체`)**: 각 Skill이 **무엇을 하는지 + 언제 쓰는지 + 언제 쓰지 않는지(경계)**를 분명하게 적습니다.

시험할 질문 네 개는 일부러 경계가 겹치도록 만들었습니다. 예를 들어 회의록 원문을 붙여 주면서 참석자에게 보낼 메일을 써 달라고 하거나, 회의 일정 변경 안내 메일을 써 달라고 요청해 두 Skill 사이에 걸치게 했습니다.

각 질문 옆에 적은 기대 Skill은 **출제자가 의도한 기대값**일 뿐 절대적인 정답표가 아닙니다. 각 조건을 3회씩 반복 실행해 질문별로 어떤 Skill이 몇 번 활성화되었는지 표로 확인해 봅시다(3회 역시 통계적 비율이라고 부르기에는 작은 횟수입니다. 만약 기대와 다른 Skill을 읽었다면 모델이 무조건 틀렸다고 단정하기 전에 질문 자체가 두 Skill 모두에 걸쳐 있지는 않은지 함께 살펴보세요).

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 70회 — 경쟁하는 Skill 셋에서 막연한 설명과 구체적인 설명을 비교합니다
# 바꿀 값: PREDICTION, 그리고 아래 DESCRIPTIONS. 먼저 예측을 적고 실행하세요: 어느 질문에서 두 설명 버전의 차이가 크게 나타날까요?
from concurrent.futures import ThreadPoolExecutor

# [1단계] 내 예측과 반복 횟수, 세 가지 회의 관련 Skill의 본문(BODIES) 정의
PREDICTION = "여기에 예측을 적습니다"
REPEATS = 3

BODIES = {
    "meeting-notes": "결정, 할 일, 미정 사항을 각각 한 줄로 쓴다.",
    "meeting-schedule": "가능한 시간 후보 두 개를 제시하고 참석자 확인을 요청한다.",
    "meeting-followup": "인사, 결정, 할 일, 기한 순서로 참석자에게 보낼 안내 메일을 쓴다.",
}

# [2단계] 비교할 두 가지 설명 버전(경계가 없는 막연 vs 무엇을·언제·경계를 적은 구체) 정의
DESCRIPTIONS = {
    "막연": {
        "meeting-notes": "회의 내용을 다룬다.",
        "meeting-schedule": "회의 일정을 다룬다.",
        "meeting-followup": "회의 후속 작업을 다룬다.",
    },
    "구체": {
        "meeting-notes": (
            "회의록 본문을 결정, 할 일, 미정 세 줄로 요약한다. "
            "회의록이 주어지고 요약이나 정리를 요청받을 때 쓴다. "
            "일정 조율이나 안내 메일 작성에는 쓰지 않는다."
        ),
        "meeting-schedule": (
            "회의 시간 후보를 정하고 참석자 일정을 맞춘다. "
            "새 회의 시간을 잡거나 옮길 때 쓴다. "
            "회의록 요약이나 메일 작성에는 쓰지 않는다."
        ),
        "meeting-followup": (
            "회의가 끝난 뒤 참석자에게 보낼 안내 메일을 쓴다. "
            "결정과 할 일을 메일 형태로 알릴 때 쓴다. "
            "회의록 요약만 원하거나 일정만 잡을 때는 쓰지 않는다."
        ),
    },
}

# [3단계] 경계가 겹치도록 설계한 시험 질문 4개와 출제자의 기대 Skill 정의
COMPETE_QUESTIONS = [
    ("다음 회의록을 세 줄로 요약해 주세요.\n" + MEETING, "meeting-notes"),
    ("다음 주 화요일에 회의를 잡고 싶어요. 가능한 시간을 정해 주세요.", "meeting-schedule"),
    ("아래 회의 결과를 참석자에게 보낼 메일로 써 주세요.\n" + MEETING, "meeting-followup"),
    ("4월 7일 회의를 4월 9일로 옮기기로 했어요. 참석자에게 알릴 메일을 써 주세요.", "meeting-followup"),
]

# [4단계] 두 버전의 Skill 폴더를 각각 생성합니다 (compete-막연, compete-구체)
for version, descriptions in DESCRIPTIONS.items():
    for name, description in descriptions.items():
        write_skill(SANDBOX / f"compete-{version}", name, description, BODIES[name])

# [5단계] 실행할 작업 목록(jobs)을 만듭니다: 설명 2가지 x 질문 4개 x 3회 = 24번의 skill_turn
announce("경쟁 Skill 셋 비교 (질문 4 x 설명 2 x 3회)", model_calls=70)
jobs = []
for version in DESCRIPTIONS:
    for question, expected in COMPETE_QUESTIONS:
        for repeat in range(REPEATS):
            jobs.append((version, question, expected, repeat))


def run_job(job):
    """작업 하나(설명 버전, 질문, 기대 Skill, 반복 번호)를 해당 버전의 Skill 폴더로 실행합니다."""
    version, question, expected, repeat = job
    skill_root = SANDBOX / f"compete-{version}"
    return skill_turn(skill_root, question)


# [6단계] 작업 24개를 동시에 최대 4개씩 실행합니다 (pool.map은 jobs와 같은 순서로 결과를 돌려줍니다)
with ThreadPoolExecutor(max_workers=4) as pool:
    turns = list(pool.map(run_job, jobs))

# [7단계] (설명 버전, 질문)별로 각 Skill이 읽힌 횟수와, 기대한 Skill만 단독으로 읽은 횟수(정확)를 집계합니다
counts = {}
for job, finished_turn in zip(jobs, turns):
    version, question, expected, repeat = job

    # 처음 보는 (버전, 질문) 조합이면 Skill별 0과 정확 0으로 시작하는 행을 만듭니다
    key = (version, question)
    if key not in counts:
        new_row = {}
        for name in BODIES:
            new_row[name] = 0
        new_row["정확"] = 0
        counts[key] = new_row
    row = counts[key]

    # 이번 실행에서 읽힌 Skill마다 1을 더합니다
    for name in BODIES:
        if activated(finished_turn, name):
            row[name] += 1

    # 읽힌 Skill이 정확히 기대한 하나뿐이면 정확에 1을 더합니다
    if set(finished_turn["seen"]) == {expected}:
        row["정확"] += 1

# [8단계] 비교 결과 표를 출력합니다
print("내 예측:", PREDICTION)
skill_short_names = []
for name in BODIES:
    skill_short_names.append(f"{name.split('-')[1]:>8s}")
header_skills = " ".join(skill_short_names)
print(f"{'질문':4s} {'기대 Skill':16s} {'설명':4s} {header_skills}   기대만 읽음")

for number, (question, expected) in enumerate(COMPETE_QUESTIONS, 1):
    for version in DESCRIPTIONS:
        row = counts[(version, question)]
        skill_cells = []
        for name in BODIES:
            skill_cells.append(f"{row[name]}/{REPEATS:>6d}")
        skill_cols = " ".join(skill_cells)
        print(f"Q{number:<3d} {expected:16s} {version:4s} {skill_cols}   {row['정확']}/{REPEATS}")

# [9단계] 설명 버전별로 기대한 Skill만 정확히 읽은 횟수를 합산합니다
totals = {}
for version in DESCRIPTIONS:
    version_total = 0
    for question, expected in COMPETE_QUESTIONS:
        version_total += counts[(version, question)]["정확"]
    totals[version] = version_total
print("합계(기대한 Skill만 정확히 읽은 횟수):", totals, "/", REPEATS * len(COMPETE_QUESTIONS), "회 중")

표에서 각 질문(Q1~Q4)마다 `막연` 줄과 `구체` 줄을 위아래로 비교해 보세요. 오른쪽 끝의 `기대만 읽음` 열이 달라졌다면, 구체적인 설명(특히 "무엇에는 쓰지 않는다"는 경계 조건)이 경쟁 상황에서 모델의 선택을 바꾼 것입니다.

만약 두 버전의 숫자가 거의 비슷하게 나왔다면 어떻게 해석해야 할까요? 모델이 막연한 설명만 보고도 Skill의 폴더 이름(`meeting-schedule`, `meeting-followup` 등)에서 힌트를 얻어 충분히 짐작했을 수 있습니다. 또한 질문 4개와 3회 반복이라는 작은 표본만으로는 두 설명 사이에 차이가 있다고도, 없다고도 단정할 수 없습니다. **차이가 크게 보이지 않았다고 해서 설명을 대충 써도 된다는 증거는 아닙니다.** 질문의 경계를 더 아슬아슬하게 만들거나 Skill 개수를 늘리면 결과가 달라질 수 있으며, 직접 `COMPETE_QUESTIONS`를 수정해 다시 실험해 볼 수 있습니다(단, 결과를 먼저 본 뒤에 차이가 잘 나는 질문만 골라 남기면 편향이 생기므로, 질문을 바꿨다면 어떤 질문을 왜 바꿨는지 함께 기록해 두는 것이 좋습니다).

## 5절. 출력 형식 더하기, 참조 파일 나누기

Skill 파일에 담기 좋은 내용은 크게 세 가지입니다.
1. **절차**: 어떤 순서로 작업할 것인가
2. **출력 형식**: 최종 답변을 어떤 모양(절 제목, 표, 목록 등)으로 작성할 것인가
3. **경계**: 어떤 경우에는 이 Skill을 쓰지 말아야 하는가

이번 절에서는 먼저 본문에 **출력 형식**을 지정해 파이썬 코드를 한 줄도 고치지 않고 답변 모양을 바꾸어 보겠습니다. 그런 다음, 형식 설명이 길어질 때를 대비해 별도의 **참조 파일(`references/format.md`)**로 분리하는 방법을 실습합니다. 각 셀은 `meeting-notes` Skill을 조금씩 변형한 복사본을 `outputs/skill_sandbox/` 아래의 서로 다른 하위 폴더에 만들어 사용합니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 3회 — 본문에 마크다운 절 제목 형식을 지시합니다
# [1단계] 본문 끝에 절 제목 형식 한 줄을 더한 Skill을 만듭니다 (파이썬 코드는 그대로입니다)
announce("절 제목 형식을 지시한 Skill을 실행합니다", model_calls=3)
FORMAT_BODY = BODY + "\n답은 `## 결정`, `## 할 일`, `## 미정` 세 절 제목으로 나눠 쓴다."
write_skill(SANDBOX / "format", "meeting-notes", STRONG_DESCRIPTION, FORMAT_BODY)

# [2단계] 같은 직접 요청 질문을 실행하고 활성화 여부와 답변을 출력합니다
turn = skill_turn(SANDBOX / "format", QUESTIONS["직접"])
print("활성화 여부:", activated(turn))
print("\n[Agent 답변]")
print(turn["answer"])

출력된 답변에 `## 결정`, `## 할 일`, `## 미정` 세 절 제목이 잘 들어갔나요? 파이썬 코드는 그대로인데 `SKILL.md` 본문에 한 줄을 더한 것만으로 답변 구조가 바뀌었습니다.

그렇다면 이번에는 출력 형식 지시를 마크다운 **표(`| 구분 | 내용 |`)**로 바꾸어 보겠습니다. 도구 호출 횟수와 답변 모양 중 무엇이 바뀔지 먼저 예측해 본 뒤 실행해 보세요.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 3회 — 출력 형식 지시를 마크다운 표로 바꿉니다
# 바꿀 값: TABLE_RULE. 예측: 앞 셀과 비교해 도구 호출 횟수와 답변 모양 중 무엇이 바뀔까요?
# [1단계] 절 제목 대신 표 형식을 지시하는 한 줄을 정합니다
TABLE_RULE = "답은 `| 구분 | 내용 |` 두 열 표로 쓴다. 행은 결정, 할 일, 미정이다."

# [2단계] 표 지시가 들어간 Skill을 만들고 같은 질문을 실행합니다
announce("출력 형식 지시를 표로 바꾸어 실행합니다", model_calls=3)
table_body = BODY + "\n" + TABLE_RULE
write_skill(SANDBOX / "table", "meeting-notes", STRONG_DESCRIPTION, table_body)
table_turn = skill_turn(SANDBOX / "table", QUESTIONS["직접"])

# [3단계] 앞 셀(절 제목 지시, turn)과 도구 호출 횟수를 비교하고 답변을 출력합니다
heading_calls = len(turn["calls"])
table_calls = len(table_turn["calls"])
print(f"도구 호출 횟수: 절 제목 지시={heading_calls}회 / 표 지시={table_calls}회")
print("읽은 파일 경로:", read_paths(table_turn))
print("\n[Agent 답변]")
print(table_turn["answer"])

모델에게 글로만 절 제목을 지시하는 것을 넘어, 애플리케이션 코드에서도 그 절 제목 기준을 검증하고 싶다면 어떻게 할까요? Skill 폴더 안에 `output.json` 파일을 두면 `skill_spec` 함수가 그 규격을 읽어 들입니다. 뒤이어 05번 노트북에서 배울 답변 검증기(`validate_answer`)가 바로 이 `output.json` 파일을 읽어서 모델이 필수 절 제목을 빠뜨리지 않았는지 코드로 검사합니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회 — 절 제목 규격을 output.json 파일로 저장하고 읽습니다
# [1단계] 절 제목 형식을 지시한 Skill 폴더를 새 위치(spec)에 만듭니다
spec_root = SANDBOX / "spec"
write_skill(spec_root, "meeting-notes", STRONG_DESCRIPTION, FORMAT_BODY)

# [2단계] 코드가 검사할 수 있도록 같은 규격을 데이터(output.json)로 선언합니다
output_spec = {
    "sections": [
        {"heading": "결정", "style": "paragraph"},
        {"heading": "할 일", "style": "steps"},
        {"heading": "미정", "style": "paragraph"},
    ]
}
spec_path = spec_root / "meeting-notes" / "output.json"
spec_text = json.dumps(output_spec, ensure_ascii=False, indent=2)
spec_path.write_text(spec_text, encoding="utf-8")

# [3단계] skill_spec 함수로 규격을 다시 읽어 절 제목과 스타일을 출력합니다
spec = skill_spec("meeting-notes", spec_root)
loaded_sections = []
for section in spec["sections"]:
    loaded_sections.append((section["heading"], section["style"]))
print("읽어 들인 절 규격:", loaded_sections)

만약 누군가 `output.json`을 편집하다가 허용되지 않은 `style` 이름(예: `"chart"`)을 잘못 적으면 어떻게 될까요? 아래 `[깨뜨리기]` 셀을 실행해 `skill_spec`이 잘못된 규격을 어떻게 잡아내는지 확인해 보세요.

In [ ]:
# [깨뜨리기] 무료, 모델 호출 0회 — output.json의 style에 허용되지 않은 값을 넣어 봅니다
# [1단계] 앞 셀에서 저장한 올바른 규격을 읽어 와, 첫 절의 style만 허용되지 않는 값("chart")으로 바꿉니다
good_spec_path = spec_root / "meeting-notes" / "output.json"
bad = json.loads(good_spec_path.read_text(encoding="utf-8"))
bad["sections"][0]["style"] = "chart"

# [2단계] 잘못된 규격을 담은 Skill 폴더를 새로 만듭니다
bad_root = SANDBOX / "spec-bad"
write_skill(bad_root, "meeting-notes", STRONG_DESCRIPTION, FORMAT_BODY)
bad_spec_path = bad_root / "meeting-notes" / "output.json"
bad_spec_path.write_text(json.dumps(bad, ensure_ascii=False), encoding="utf-8")

# [3단계] skill_spec이 모델을 호출하기도 전에 ValueError로 잡아내는지 확인합니다
try:
    skill_spec("meeting-notes", bad_root)
except ValueError as error:
    print("잡아낸 ValueError:", error)

위 출력에서 볼 수 있듯, `output.json`에서 허용하는 `style` 값은 `table`, `steps`, `paragraph` 세 가지뿐입니다. 이처럼 출력 형식 규격을 별도 데이터 파일로 선언해 두면, 잘못된 설정값이 들어왔을 때 모델을 호출하기도 전에 파이썬 코드가 먼저 오류를 잡아낼 수 있습니다.

이제 상세한 출력 형식 설명을 **참조 파일(`references/format.md`)**로 분리해 보겠습니다. `SKILL.md` 본문은 핵심 절차만 짧게 남겨 두고, "출력 형식은 `/meeting-notes/references/format.md`를 읽고 그대로 따른다"고만 안내합니다. 이것이 점진적 공개의 세 번째 단계입니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 3회 — 상세 형식 설명을 references/format.md로 분리합니다
# [1단계] SKILL.md 본문은 짧게 두고, 형식 설명은 참조 파일(references/format.md)에 둡니다
announce("참조 파일로 나눈 Skill을 실행합니다", model_calls=3)
split_body = BODY + "\n출력 형식은 /meeting-notes/references/format.md를 읽고 그대로 따른다."
format_reference = "답은 `## 결정`, `## 할 일`, `## 미정` 세 절 제목으로 나눠 쓴다.\n"
write_skill(
    SANDBOX / "split",
    "meeting-notes",
    STRONG_DESCRIPTION,
    split_body,
    files={"references/format.md": format_reference},
)

# [2단계] 같은 질문을 실행하고 읽은 파일의 순서와 활성화된 Skill을 확인합니다
split_turn = skill_turn(SANDBOX / "split", QUESTIONS["직접"])
print("읽은 파일 경로(순서대로):", read_paths(split_turn))
print("활성화된 Skill 목록:", split_turn["seen"])
print("\n[Agent 답변]")
print(split_turn["answer"])

출력된 결과를 통해 다음 두 가지를 확인해 보세요.
- 모델이 읽은 파일 경로가 `/meeting-notes/SKILL.md`에서 `/meeting-notes/references/format.md`로 순서대로 이어졌나요?
- 활성화된 Skill 목록(`seen`)에는 참조 파일이 아니라 `meeting-notes`(`SKILL.md`)만 기록됩니다.

앞서 활성화의 정의는 "`SKILL.md`를 **끝까지** 읽는 것"이라고 했습니다. 만약 모델이 파일의 일부만 읽으면 어떻게 처리될까요? 아래 `[바꿔보기]` 셀에서는 한 번에 읽는 줄 수(`limit`)를 1000줄에서 3줄로 줄여 봅니다. 모델을 호출하지 않고 백엔드의 파일 읽기 동작만 직접 시험합니다.

In [ ]:
# [바꿔보기] 무료, 모델 호출 0회 — 파일 읽기 줄 수(limit)를 1000에서 3으로 줄여 봅니다
# 바꿀 값: LIMIT. 예측: 앞부분 3줄만 읽었을 때도 활성화(seen)로 기록될까요?
LIMIT = 3

# 줄 수를 1000(파일 전체)과 LIMIT(앞부분만)로 바꿔 가며 백엔드의 파일 읽기만 직접 시험합니다
for limit in (1000, LIMIT):
    # [1단계] 활성화 기록용 목록과 추적 백엔드를 새로 만듭니다
    seen = []
    backend = TrackedSkillsBackend(SANDBOX / "format", seen.append)

    # [2단계] 0번째 줄부터 limit줄만 읽습니다 (next_offset이 None이면 끝까지 읽은 것입니다)
    result = backend.read("/meeting-notes/SKILL.md", 0, limit)
    print(f"limit={limit:4d} -> 활성화 기록={seen}, next_offset={result.next_offset}")

반환된 `result.next_offset`이 `None`이 아니라면 파일 뒤에 아직 읽지 않은 줄이 남아 있다는 뜻입니다. `TrackedSkillsBackend`는 `offset == 0`이고 `next_offset is None`일 때만(즉, 처음부터 끝까지 다 읽었을 때만) 활성화된 것으로 기록합니다. 절차 문서의 앞부분 몇 줄만 읽고 만 것은 절차를 제대로 읽은 것이 아니기 때문입니다.

## 6절. 트리거 테스트와 near-miss

Skill을 잘 만들었는지 확인하려면 **트리거 테스트(Trigger Test)**를 설계해야 합니다. 트리거 테스트에서는 시험할 질문을 세 가지 범주로 나눕니다.

- **positive (양성 질문)**: 이 Skill을 반드시 읽어야 하는 질문
- **near-miss (경계 부정 질문)**: 질문에 등장하는 핵심 단어는 겹치지만, 실제로는 이 Skill을 읽으면 안 되는 질문
- **far (무관한 부정 질문)**: 주제와 단어가 모두 전혀 관련 없는 질문

| 종류 | 질문 예시 | 기대 동작 |
|---|---|---|
| **positive** | "다음 회의록을 요약해 주세요" | 읽어야 함 (`True`) |
| **near-miss** | "다음 주 회의 일정을 잡아 주세요" | 읽으면 안 됨 (`False`) |
| **far** | "오늘 날씨는 어때?" | 읽으면 안 됨 (`False`) |

여기서 우리가 세어야 할 오류는 두 가지입니다.
- **위양성(False Positive)**: 읽으면 안 되는 질문(near-miss, far)인데 읽은 경우
- **위음성(False Negative)**: 읽어야 하는 질문(positive)인데 읽지 않은 경우

질문마다 한 번씩만 돌려 보면 1회 관찰이고, 여러 번 반복 실행해 비율을 구해야 비로소 신뢰할 수 있는 지표가 됩니다.

## 7절. 읽으면 안 되는 질문

예를 들어 "다음 주 회의 일정을 잡아 주세요"라는 질문에는 `회의`라는 단어가 들어 있지만, 회의록을 요약해 달라는 요청이 아니라 일정을 잡아 달라는 요청입니다. 따라서 `meeting-notes` Skill을 읽으면 안 되는 대표적인 **near-miss** 질문입니다. 먼저 이 질문을 던졌을 때 Agent가 어떻게 반응하는지 확인해 보겠습니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 3회 — 단어('회의')는 겹치지만 읽으면 안 되는 near-miss 질문을 던집니다
# [1단계] near-miss 질문을 정하고, 4절에서 만든 구체적 설명 Skill(desc-strong)로 실행합니다
NEAR_MISS = "다음 주 회의 일정을 잡아 주세요."
announce("near-miss 질문 하나를 실행합니다", model_calls=3)
near = skill_turn(SANDBOX / "desc-strong", NEAR_MISS)

# [2단계] 읽으면 안 되는 질문을 읽었는지(위양성) 확인하고 답변을 출력합니다
print("활성화 여부:", activated(near), "| 읽은 파일 경로:", read_paths(near))
print("\n[Agent 답변]")
print(near["answer"])

위 셀에서 `활성화 여부`가 어떻게 나왔나요? 이미 `False`가 나왔다면 모델이 현재 설명만 보고도 요약 요청이 아님을 잘 구분한 것이고, `True`가 나왔다면 `회의`라는 단어에 이끌려 불필요하게 파일을 열어 본(위양성) 것입니다.

이 경계를 더 분명하게 그어 주기 위해, 아래 `[바꿔보기]` 셀에서는 `description` 끝에 `"회의 일정을 잡을 때는 쓰지 않는다."`라는 경계 문장을 덧붙여 다시 실행해 보겠습니다.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 3회 — description 끝에 쓰지 말아야 할 경계 조건을 덧붙입니다
# 바꿀 값: BOUNDARY. 예측: 활성화 여부가 바뀔까요? (앞 셀에서 이미 안 읽었다면 결과는 같게 나옵니다.)
# [1단계] 설명 끝에 덧붙일 경계 문장을 정하고, 덧붙인 설명으로 Skill을 만듭니다
BOUNDARY = " 회의 일정을 잡을 때는 쓰지 않는다."
announce("경계 조건을 더한 설명으로 near-miss 질문을 다시 실행합니다", model_calls=3)
boundary_description = STRONG_DESCRIPTION + BOUNDARY
write_skill(SANDBOX / "boundary", "meeting-notes", boundary_description, BODY)

# [2단계] 같은 near-miss 질문을 다시 실행해 경계를 추가하기 전(near)과 후(near_after)를 비교합니다
near_after = skill_turn(SANDBOX / "boundary", NEAR_MISS)
print("경계 추가 전 활성화:", activated(near), "-> 경계 추가 후 활성화:", activated(near_after))
print("읽은 파일 경로:", read_paths(near_after))

이제 경계 조건을 추가한 `boundary` Skill을 대상으로, 읽어야 하는 질문(positive) 2개와 읽으면 안 되는 질문(near-miss) 2개를 묶어 총 4문항짜리 미니 트리거 테스트를 돌려 보겠습니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 12회 — 가장 작은 규모의 트리거 테스트(질문 4개, 각 1회)를 실행합니다
# [1단계] (질문, 읽어야 하는가) 4문항을 정합니다: 양성 2개, near-miss 2개
CHECKS = [
    ("다음 회의록을 요약해 주세요.\n" + MEETING, True),
    ("오늘 미팅 내용을 결정과 할 일로 정리해 줘.\n" + MEETING, True),
    ("다음 주 회의 일정을 잡아 주세요.", False),
    ("회의실 예약은 어떻게 하나요?", False),
]

# [2단계] 위양성(읽으면 안 되는데 읽음)과 위음성(읽어야 하는데 놓침)을 담을 목록을 준비합니다
announce("트리거 테스트 4문항을 실행합니다", model_calls=12)
false_positive = []
false_negative = []

# [3단계] 문항마다 경계 조건이 있는 Skill(boundary)로 실행하고 기대와 비교합니다
for question, should in CHECKS:
    turn_result = skill_turn(SANDBOX / "boundary", question)
    got = activated(turn_result)

    verdict = "통과" if got == should else "실패"
    first_line = question.splitlines()[0]
    short_q = first_line[:28]
    print(f"[{verdict}] 읽어야 함={str(should):5s} | 실제 읽음={str(got):5s} :: {short_q}")

    if got and not should:
        false_positive.append(question)
    if should and not got:
        false_negative.append(question)

# [4단계] 두 종류의 오류 건수를 집계해 출력합니다
print(f"\n집계 -> 위양성(불필요하게 읽음): {len(false_positive)}건 / 위음성(놓침): {len(false_negative)}건")

위 셀은 질문 네 개를 각각 한 번씩만 실행해 본 **가장 작은 형태의 트리거 테스트**입니다. 네 문항 모두 `통과`가 나왔더라도 "이 Skill이 완전히 안정적이다"라고 결론 내릴 수는 없으며, "이번 네 번의 관찰에서는 기대대로 동작했다"고 읽어야 합니다. 20개 질문을 3회씩 반복하는 전체 측정 코드는 노트북 맨 아래의 선택 측정 절에 준비되어 있습니다.

이제 저장소에 기본 제공되는 업무용 Skill(`skills/`)을 사용해 실제 RAG 질문을 하나 실행해 보겠습니다. 검색기(`Retriever`)와 검색 범위(`Scope`)는 바로 아래 셀에서 생성합니다. 범위 필터, 근거(Evidence) ID, 인용 검증의 상세한 원리는 04번과 05번 노트북에서 깊이 있게 다룹니다. 여기서는 Agent가 어떤 Skill을 읽었는지(`loaded_skills`)와 최종 상태(`status`)만 먼저 확인하고, 만약 검증 오류가 발생하면 아래 해설을 따라 왜 거부되었는지 읽어 보세요. 이 실습이 05번 노트북의 운영 검증으로 이어지는 다리 역할을 합니다.

먼저 `skills/` 폴더 안에 어떤 업무용 Skill들이 준비되어 있고 각각의 설명과 출력 규격(`output.json`)이 어떻게 생겼는지 무료 코드 셀로 살펴봅시다.


In [ ]:
# [따라하기] 무료, 모델 호출 0회 — 저장소에 기본 제공되는 6개 업무용 Skill의 설명과 절 제목 규격 살펴보기
import re

for folder in sorted(SKILLS.iterdir()):
    # [1단계] SKILL.md가 없는 항목은 건너뜁니다
    skill_file = folder / "SKILL.md"
    if not skill_file.is_file():
        continue

    # [2단계] SKILL.md에서 description 줄을 찾아 설명 문장을 꺼냅니다
    raw = skill_file.read_text(encoding="utf-8")
    desc_match = re.search(r"description:\s*(.+)", raw)
    if desc_match:
        desc = desc_match.group(1).strip()
    else:
        desc = ""

    # [3단계] output.json이 있으면 답변용 Skill이므로 필수 절 제목을 읽고, 없으면 검색 안내용으로 표시합니다
    spec_file = folder / "output.json"
    if spec_file.is_file():
        spec = skill_spec(folder.name, SKILLS)
        headings = []
        for section in spec["sections"]:
            headings.append(section["heading"])
        role_note = f"답변용 Skill | 필수 절 제목: {headings}"
    else:
        role_note = "검색 안내용 Skill (output.json 없음)"

    # [4단계] 이름, 역할, 설명(앞 90자)을 출력합니다
    print(f"[{folder.name}] ({role_note})")
    print(f"  설명: {desc[:90]}...")

**관찰할 포인트:**
- `skills/` 폴더에는 총 6개의 Skill이 있습니다. 그중 `grounded-answer`는 검색 전략을 안내하는 보조 Skill이고, 나머지 다섯 개(`comparison`, `grounded-qa`, `insufficient-evidence`, `procedure`, `table-analysis`)는 질문 성격에 따라 골라 쓰는 **답변용 Skill**입니다.
- 이제 아래 셀에서 `"알파 SLA 표의 서비스 크레딧 비율 중 최댓값은 무엇인가요?"`라는 질문을 던졌을 때, 에이전트가 이 중 어느 Skill을 골라 읽고 답변을 작성하는지 확인해 보세요.

In [ ]:
# [따라하기] 유료, 모델 호출 약 3회 — 기본 제공 Skill(skills/)로 실제 SLA 질문을 실행합니다
# [1단계] 실행 직전에 규모를 알리고, 검색기(Retriever)와 검색 범위(고객 알파, 기준일 2026-09-08)를 준비합니다
announce("기본 제공 Skill로 RAG 질문을 한 번 실행합니다", model_calls=3)
retriever = Retriever(load_corpus())
scope = Scope("알파", date(2026, 9, 8))

# [2단계] 질문을 실행합니다. max_attempts=1이므로 검증에 실패해도 재시도하지 않습니다
record = ask(
    "알파 SLA 표의 서비스 크레딧 비율 중 최댓값은 무엇인가요?",
    retriever=retriever,
    scope=scope,
    model=make_model(),
    max_attempts=1,
)

# [3단계] 상태, 검증 결과, 오류 목록, 활성화된 Skill을 출력합니다
print("상태(status):", record["status"])
print("검증(validation):", record["validation"])
print("오류 목록(errors):", record["errors"])
print("활성화된 Skill(loaded_skills):", record["loaded_skills"])
print()

# [4단계] 답변 문장마다 어떤 근거 ID에 연결했는지 출력합니다
answer = record["answer"] or {}
for section in answer.get("sections", []):
    for item in section["items"]:
        print("답변 문장:", item["text"], "| 연결한 근거 ID:", item["evidence_ids"])

# [5단계] 추적 기록 중 Skill을 읽은 이벤트(skill_read)만 골라 출력합니다
skill_read_events = []
for event in record["trace"]["events"]:
    if event["event"] == "skill_read":
        skill_read_events.append(event)
print("skill_read 이벤트:", skill_read_events)

위 셀에서 `status`가 `error`로 나오더라도 놀라지 마세요. 만약 `errors`에 `인용문에 없는 숫자: …`라는 메시지가 보인다면, 먼저 **어느 답변 문장의 어떤 숫자 때문에 거부되었는지** 찾아보세요. 답변 검증기(`validate_answer`)는 답변 문장에 적힌 모든 숫자가 **그 문장에 직접 연결한(`evidence_ids`) 인용문 원문 안에 실제로 들어 있는지**를 엄격하게 대조합니다.

이전에 저자가 이 셀의 질문을 따로 9번 반복 실행해 관찰했을 때 7번은 통과했고 2번은 거부되었습니다(`인용문에 없는 숫자: 20%`). 아래 설명은 그 이전 관찰을 정리한 것이며, 여러분이 방금 실행한 결과나 2026-11-02 실행 기록과는 별개입니다.
- **통과한 경우**: SLA 표의 내용을 옮긴 문장이 `20%`가 들어 있는 SLA 표 근거에 제대로 연결되어 있었습니다.
- **거부된 경우**: 모델이 `"추가 약정 A-01은 신청 기한만 바꾸므로 최댓값은 여전히 20%입니다"`라고 맞는 설명을 쓰면서, 정작 근거 ID는 **추가 약정 문서에만** 연결해 두었습니다. 추가 약정 문서 원문에는 `20%`라는 숫자가 적혀 있지 않습니다. 문장의 뜻이 사실과 맞더라도, 그 숫자가 적힌 SLA 표 근거를 해당 문장에 함께 연결하지 않으면 검증 코드는 출처를 확인할 수 없으므로 정당하게 거부한 것입니다(표를 읽고 사칙연산으로 새로 계산해 낸 파생 숫자도 같은 이유로 거부됩니다).

참고로 2026-11-02 실행 기록의 1회 실행에서는 이 셀이 `status`가 `error`로 끝났고 `errors`에 `인용문에 없는 숫자: 20%`가 나왔습니다. 그 실행에서 거부된 문장은 최댓값 `20%`를 말하면서 근거 ID를 추가 약정 문서(`E-e4d275a833de`)와 다른 근거에만 연결하고 SLA 표 근거(`E-18da4c227703`)는 연결하지 않은 문장이었습니다. 다만 이것도 1회 실행의 기록일 뿐 여러분의 실행에서 같은 오류가 나온다는 보장은 없습니다.

모델은 실행할 때마다 조금씩 다른 문장과 인용 구성을 내놓으므로, 여러분의 실행 결과에서 어떤 문장이 어떤 근거 ID에 연결되었는지 직접 대조해 보세요. 여기서 꼭 구분해야 할 점은 두 가지입니다.
1. `loaded_skills` 기록은 모델이 Skill 문서를 끝까지 읽었다는 증거입니다.
2. `status`와 `validation`은 모델이 작성한 답변이 코드 수준의 인용 검증을 통과했는지를 따로 보여 줍니다. **Skill을 잘 읽었다고 해서 답변의 인용이 자동으로 정확해지는 것은 아닙니다.** 검증에 실패한 구체적인 이유는 `errors`에 그대로 남습니다.

앞 셀에서 검증 오류의 원인을 확인했다면 어떻게 해결해야 할까요? 질문을 쉬운 것으로 바꾸거나 검증 규칙을 느슨하게 푸는 것은 올바른 해법이 아닙니다. 필요한 숫자는 이미 원문에 들어 있고, 문제는 모델이 문장과 근거 ID를 잘못 짝지은 데 있기 때문입니다.

아래 셀은 **똑같은 질문**을 `ask`의 기본 설정인 `max_attempts=2`로 실행합니다. 첫 번째 답변이 검증에 실패하면, 코드가 검증 오류 메시지를 모델에게 그대로 돌려주어 인용 연결을 고칠 기회를 한 번 더 줍니다. 검증 기준과 예산(`Budget`) 한도는 그대로 유지되며, 재시도에 쓰인 모델 호출도 같은 예산 안에서 엄격하게 합산됩니다.

재시도는 첫 번째 답변이 검증에 실패했을 때만 일어납니다. 따라서 아래 셀에서 `검증 실패 후 재시도한 횟수`가 `0`으로 나온다면, 재시도가 효과를 낸 것이 아니라 이번 실행에서는 첫 번째 답변이 한 번에 검증을 통과했다는 뜻입니다(이전에 저자가 따로 한 13회 실험에서는 모두 최종 통과했고, 그중 5회는 첫 답변이 거부된 뒤 재시도로 통과했습니다. 물론 표본이 작으므로 100% 통과를 보장한다는 뜻은 아닙니다. 2026-11-02 실행 기록의 1회 실행에서는 재시도한 횟수가 `0`이었고 첫 답변이 한 번에 통과했습니다. 앞 셀은 오류로 끝난 같은 질문이었지만, 이 셀은 `ask`를 처음부터 새로 실행하므로 앞 셀의 실패가 이어지지 않습니다). 만약 두 번째 시도까지 실패하면 최종 `status`는 `error`로 끝나고, 어느 문장이 문제였는지 그대로 출력합니다.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 3~7회 — 질문은 그대로 두고 검증 오류를 알려 주는 재시도(max_attempts=2)를 켭니다
# 바꿀 값: MAX_ATTEMPTS. 예측: 앞 셀에서 났던 오류가 다시 나올까요? 재시도는 몇 번 일어날까요? (1로 바꾸면 앞 셀과 같은 조건이 됩니다.)
# [1단계] 같은 질문을 재시도 허용 횟수만 바꿔 다시 실행합니다
MAX_ATTEMPTS = 2
announce("같은 질문에 대해 검증 오류를 돌려주고 재시도를 허용해 실행합니다", model_calls=5)
fixed = ask(
    record["question"],
    retriever=retriever,
    scope=scope,
    model=make_model(),
    max_attempts=MAX_ATTEMPTS,
)

# [2단계] 추적 기록에서 검증 실패로 재시도가 일어난 이벤트를 찾아 첫 번째 오류 메시지만 모읍니다
failed_tries = []
for event in fixed["trace"]["events"]:
    if event["event"] == "validation_retry":
        failed_tries.append(event["errors"][0])

# [3단계] 앞 셀(재시도 없음)과 이번 셀(재시도 허용)의 결과를 나란히 출력합니다
print("앞 셀 결과(재시도 없음):", record["status"], record["errors"][:1])
print("이번 실행에서 검증 실패 후 재시도한 횟수:", len(failed_tries), failed_tries)
print("이번 최종 결과:", fixed["status"], "| 검증:", fixed["validation"], "| 오류:", fixed["errors"])

# [4단계] 재시도 끝에도 실패했다면 어느 답변 문장이 문제였는지 보여 줍니다
if fixed["status"] == "error":
    fixed_answer = fixed["answer"] or {}
    for section in fixed_answer.get("sections", []):
        for item in section["items"]:
            print("답변 문장:", item["text"], "| 연결한 근거 ID:", item["evidence_ids"])

# [5단계] 렌더링된 최종 답변을 출력합니다
print("\n[렌더링된 최종 답변]")
print(fixed["markdown"])

위 실행에서 `max_attempts=2`로 재시도를 허용했을 때 검증 통과 여부와 `validation_retry` 이벤트를 확인하셨나요?

## 8절. 출력 평가, Skill이 맡지 않는 것

트리거 테스트가 "모델이 Skill을 제때 읽었는가"를 확인한다면, **출력 평가(Output Evaluation)**는 "Skill을 읽은 덕분에 최종 답변이 실제로 더 나아졌는가"를 확인합니다. 같은 질문을 Skill이 있을 때(`SKILLS`)와 없을 때(빈 폴더 `no-skills`) 각각 실행하여 답변 검증 통과 여부, 읽은 Skill 목록, 모델 호출 횟수를 직접 비교해 봅시다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 6회 — 같은 질문을 Skill이 있을 때와 없을 때(빈 폴더) 각각 실행해 출력 평가 비교하기
# [1단계] Skill이 하나도 없는 빈 폴더와 평가용 질문을 준비합니다
announce("Skill이 있을 때와 없을 때의 RAG 답변을 각각 1회 실행해 비교합니다", model_calls=6)
none_dir = SANDBOX / "no-skills"
none_dir.mkdir(exist_ok=True)
eval_q = "알파 계약의 서비스 크레딧 신청 기한은 언제인가요?"

# [2단계] 비교할 두 조건: (이름표, Skill 폴더)
conditions = (
    ("Skill 있음 (skills/)", SKILLS),
    ("Skill 없음 (빈 폴더)", none_dir),
)

# [3단계] 조건마다 같은 질문을 실행하고 상태, 검증, 읽은 Skill, 모델 호출 수를 출력합니다
for label, s_dir in conditions:
    rec = ask(
        eval_q,
        retriever=retriever,
        scope=scope,
        model=make_model(),
        max_attempts=1,
        skills_dir=s_dir,
    )
    trace_counts = rec["trace"]["counts"]
    model_call_count = trace_counts.get("model", 0)
    print(
        f"[{label:18s}] 상태={rec['status']:12s} | 검증={rec['validation']:6s} "
        f"| 읽은 Skill={rec['loaded_skills']} | 모델호출={model_call_count}회"
    )
    if rec["errors"]:
        print("   -> 거부 사유(errors):", rec["errors"])

**관찰할 포인트:**
- **Skill이 있을 때(`skills/`)**: 에이전트가 질문에 맞는 답변용 Skill(이 질문에서는 보통 `grounded-qa`)을 `read_file`로 끝까지 읽었는지 `읽은 Skill` 칸에서 확인하세요. 읽었다면 `output.json`이 요구하는 절 제목(`grounded-qa`는 `결론`, `설명`)에 맞춰 답변을 작성하고 검증을 통과했는지 `상태`와 `검증` 칸에서 확인합니다. 2026-11-02 실행 기록의 1회 실행에서는 `grounded-qa`를 읽고 `answered`, `passed`로 끝났습니다.
- **Skill이 없을 때(빈 폴더)**: 에이전트가 읽을 수 있는 `SKILL.md`가 하나도 없으므로 `loaded_skills`가 비어 있고, 설령 모델이 `AnswerPayload` 스키마를 흉내 내더라도 검증기(`validate_answer`)가 Skill 본문을 읽지 않았다고 보고 거부(`status: error`)합니다. 거부 사유(`errors`)는 `거부 사유` 줄에 그대로 출력되니 직접 읽어 보세요. 2026-11-02 실행 기록의 1회 실행에서는 첫 사유가 `answer/SKILL.md 전체 본문을 read_file로 읽어야 합니다.`였고, 모델이 존재하지 않는 `answer`라는 이름을 지어내 `알 수 없는 답변 Skill입니다: answer`도 함께 나왔습니다. 즉 Skill 없음 조건의 실패는 모델이 답을 못 써서가 아니라, 검증기가 Skill 본문 읽기를 필수로 요구하기 때문에 생기는 실패입니다.

반대로 **Skill에게 맡기면 안 되는 일**도 명확히 알아 두어야 합니다.
- 호출 횟수나 비용 한도(예산, `budget`)는 파이썬 코드가 강제로 막아야 합니다.
- 어떤 고객(`entity`)과 기준일(`as_of`)의 문서만 볼 수 있는지는 검색 도구가 점수 계산 전에 미리 걸러 주어야 합니다.
- 답변이 인용한 근거가 실제 원문에 있는지는 검증 코드가 대조해야 합니다.

Skill 본문에 "절대로 예산을 넘기지 마라"거나 "다른 고객 문서를 인용하지 마라"고 아무리 적어 두어도 강제력이 없습니다. 역할을 나누자면, **Skill은 어떻게 일할지(작업 순서, 출력 형식, 선택 경계)를 안내하고, 애플리케이션 코드는 넘으면 안 되는 안전선(예산, 범위 필터, 근거 검증)을 강제합니다.** 05번 노트북에서 이 두 가지를 하나의 운영형 Agent로 결합합니다.

## 직접 해보기

내 업무에서 자주 반복되는 요약이나 문서 정리 규칙 하나를 골라 나만의 Skill로 만들어 봅시다. 아래 세 개의 `[내 것 만들기]` 셀에는 **주간 보고 요약(`weekly-report`)** 예시가 미리 채워져 있어 위에서 아래로 바로 실행해 볼 수 있습니다. 먼저 그대로 실행해 흐름을 확인한 뒤, `MY_`로 시작하는 변수 값을 내 업무 주제로 바꾸고 다시 실행해 보세요(값을 비워 두면 안내 메시지만 출력되고 모델을 호출하지 않습니다).

먼저 **1단계**로, 내 Skill의 이름(`MY_NAME`), 설명(`MY_DESCRIPTION`), 본문(`MY_BODY`)을 파일로 저장하고 `check_skill_folder`로 규칙에 맞는지 검사합니다.

In [ ]:
# [내 것 만들기] 무료, 모델 호출 0회 — 내 업무 Skill을 작성하고 규칙을 검사합니다
# [1단계] 내 Skill의 이름(소문자·숫자·하이픈), 설명(무엇을 + 언제), 본문(절차)을 정합니다
MY_NAME = "weekly-report"
MY_DESCRIPTION = "한 주 업무 메모를 완료, 진행 중, 막힘 세 줄로 요약한다. 주간 보고 요약을 요청받을 때 쓴다."
MY_BODY = "완료, 진행 중, 막힘을 각각 한 줄로 쓴다. 막힘이 없으면 '없음'이라고 쓴다."

# [2단계] 다시 실행해도 이전 결과가 남지 않도록 내 Skill 폴더를 비웁니다
MINE = SANDBOX / "mine"
if MINE.exists():
    shutil.rmtree(MINE)

# [3단계] 세 값이 모두 채워졌을 때만 파일을 쓰고 규칙 검사기로 확인합니다
all_filled = MY_NAME.strip() and MY_DESCRIPTION.strip() and MY_BODY.strip()
if not all_filled:
    print("MY_NAME(소문자·숫자·하이픈), MY_DESCRIPTION(무엇을 + 언제), MY_BODY(절차 한두 줄)를 모두 채워 주세요.")
else:
    write_skill(MINE, MY_NAME.strip(), MY_DESCRIPTION.strip(), MY_BODY.strip())
    problems = check_skill_folder(MINE / MY_NAME.strip())
    print("규칙 검사 결과:", problems or "문제 없음")

규칙 검사를 통과했다면, **2단계**로 이 Skill을 반드시 읽어야 하는 양성 질문(`MY_QUESTION`)을 던져 실제로 활성화되는지, 그리고 답변이 내가 정한 형식대로 나오는지 확인해 보세요.

In [ ]:
# [내 것 만들기] 유료, 모델 호출 약 3회 — 내 Skill을 반드시 읽어야 하는 질문(positive)을 시험합니다
# [1단계] 반드시 읽어야 하는 질문을 정합니다
MY_QUESTION = (
    "이번 주 메모를 주간 보고로 요약해 줘. "
    "신규 견적 3건 발송 완료, 보고서 양식 개편 진행 중, 승인권자 휴가로 계약서 서명 대기."
)

# [2단계] 질문이 비어 있거나 앞 셀에서 Skill 파일을 만들지 않았다면 안내만 출력하고 모델은 부르지 않습니다
my_skill_file = MINE / MY_NAME.strip() / "SKILL.md"
if not MY_QUESTION.strip() or not my_skill_file.is_file():
    print("앞 셀에서 Skill을 먼저 만들고, MY_QUESTION에 읽어야 하는 질문을 적어 주세요.")
else:
    # [3단계] 내 Skill로 질문을 실행하고 활성화 여부와 답변을 출력합니다
    announce("내 Skill에 읽어야 하는 양성 질문을 보냅니다", model_calls=3)
    mine = skill_turn(MINE, MY_QUESTION)
    is_active = MY_NAME.strip() in mine["seen"]
    print("활성화 여부:", is_active, "| 읽은 파일 경로:", read_paths(mine))
    print("\n[Agent 답변]")
    print(mine["answer"])

마지막 **3단계**로, 단어는 겹치지만 이 Skill을 읽으면 안 되는 경계 부정 질문(`MY_NEAR_MISS`)을 던져 보겠습니다. 만약 읽으면 안 되는데 읽었다면(`활성화 여부: True`), 1단계 셀의 `MY_DESCRIPTION` 끝에 `'...할 때는 쓰지 않는다'`는 경계 문장을 덧붙인 뒤 다시 실행해 보세요.

In [ ]:
# [내 것 만들기] 유료, 모델 호출 약 3회 — 단어는 겹치지만 읽으면 안 되는 질문(near-miss)을 시험합니다
# [1단계] 단어는 겹치지만 읽으면 안 되는 질문을 정합니다
MY_NEAR_MISS = "다음 주 주간 회의 시간을 목요일 오후로 옮겨 주세요."

# [2단계] 질문이 비어 있거나 Skill 파일이 없다면 안내만 출력하고 모델은 부르지 않습니다
my_skill_file = MINE / MY_NAME.strip() / "SKILL.md"
if not MY_NEAR_MISS.strip() or not my_skill_file.is_file():
    print("MY_NEAR_MISS에 핵심 단어는 겹치지만 이 Skill을 읽으면 안 되는 질문을 적어 주세요.")
else:
    # [3단계] 내 Skill로 질문을 실행하고 읽었는지(위양성) 확인합니다
    announce("내 Skill에 읽으면 안 되는 near-miss 질문을 보냅니다", model_calls=3)
    mine_near = skill_turn(MINE, MY_NEAR_MISS)
    got = MY_NAME.strip() in mine_near["seen"]
    print("활성화 여부:", got, "| 읽은 파일 경로:", read_paths(mine_near))

    # [4단계] 읽었다면 설명에 경계 문장을 덧붙이도록 안내합니다
    if got:
        print("-> 읽으면 안 되는 질문을 읽었습니다(위양성). MY_DESCRIPTION 끝에 '...할 때는 쓰지 않는다'는 경계를 덧붙여 다시 실행해 보세요.")

## 선택 측정

아래 두 코드 셀에는 표지가 붙어 있지 않으며, 맨 위 설정 셀의 `RUN_OPTIONAL = True`로 바꿨을 때만 실제로 모델을 호출합니다. 60분 수업 시간 안에 다 기다리기에는 호출 규모가 커서 **수업 중에는 기본적으로 실행하지 않도록 막아 둔 선택 측정**입니다. `RUN_OPTIONAL = False`인 상태에서는 실행하지 않았다는 안내 문구만 출력합니다.

첫 번째 선택 측정은 양성 질문 6개, near-miss 6개, 무관한 질문(far) 8개를 합친 총 20개 질문을 각각 3회씩 반복 실행하여 질문별 활성화 비율을 구하는 전체 트리거 측정입니다.

2026-11-02 실행 기록의 1회 실행(`RUN_OPTIONAL = True`)에서는 양성 질문 6개가 모두 3회 중 3회 읽혀 18회 중 18회(비율 1.00)였고, 읽으면 안 되는 질문 14개(near-miss 6개와 far 8개)는 모두 3회 중 0회 읽혀 42회 중 0회(비율 0.00)였습니다. 이 셀은 모델 호출 79회, 253.2초가 걸렸습니다. 이 값은 한 대의 컴퓨터에서 한 번 실행한 기록이므로, 여러분의 실행에서 비율이 같게 나온다는 보장은 없습니다. 같은 질문 3회가 서로 독립인지도 확인하지 않았습니다. 여러분의 표에서는 비율이 1.00이나 0.00이 아닌 질문이 있는지, 있다면 어떤 단어 때문인지를 먼저 찾아보세요.

In [ ]:
# 선택 측정 1: 전체 트리거 측정 (RUN_OPTIONAL = True일 때만 실행됩니다)
if not RUN_OPTIONAL:
    print("전체 트리거 측정(질문 20개 x 3회, 2026-11-02 실행에서 모델 호출 79회)은 이번에는 돌리지 않았습니다.")
else:
    # [1단계] 양성 6개, near-miss 6개, 무관한 질문(far) 8개와 읽어야 하는지 여부를 정의합니다
    TRIGGER_SET = [
        (f"다음 회의록을 요약해 주세요.\n{MEETING}", True),
        (f"미팅 내용 정리해 줘.\n{MEETING}", True),
        (f"이 회의 기록에서 할 일만 뽑아 줘.\n{MEETING}", True),
        (f"회의록 세 줄 요약 부탁해요.\n{MEETING}", True),
        (f"아래 회의에서 정해진 것이 뭐야?\n{MEETING}", True),
        (f"회의 결과를 짧게 보고서용으로 써 줘.\n{MEETING}", True),
        ("다음 주 회의 일정을 잡아 주세요.", False),
        ("회의실 예약은 어떻게 하나요?", False),
        ("회의 시간을 오후로 옮겨 주세요.", False),
        ("회의 참석자에게 초대장을 보내 줘.", False),
        ("회의 자료 템플릿을 만들어 줘.", False),
        ("회의 중 쓸 영어 표현을 알려 줘.", False),
        ("파이썬 리스트를 정렬하는 법은?", False),
        ("오늘 날씨는 어때?", False),
        ("점심 메뉴를 추천해 줘.", False),
        ("2의 10제곱은?", False),
        ("서울에서 부산까지 거리는?", False),
        ("이메일 제목을 지어 줘.", False),
        ("SQL JOIN을 설명해 줘.", False),
        ("한국의 수도는?", False),
    ]

    # [2단계] 질문마다 3번씩 실행해 활성화된 횟수의 비율을 구합니다
    announce("전체 트리거 측정(20문항 x 3회)을 실행합니다", model_calls=80)
    for question, should in TRIGGER_SET:
        runs = []
        for _ in range(3):
            trigger_turn = skill_turn(SANDBOX / "boundary", question)
            runs.append(activated(trigger_turn))

        rate = sum(runs) / len(runs)
        short_q = question.splitlines()[0][:24]
        print(f"읽어야 함={str(should):5s} | 활성화 비율={rate:.2f} ({len(runs)}회) :: {short_q}")

두 번째 선택 측정은 8절에서 설명한 **출력 평가(Output Evaluation)**입니다. 똑같은 SLA 질문 3개를 기본 Skill 폴더(`skills/`)가 있을 때와 빈 폴더(`no-skills`)일 때 각각 실행하여 검증 통과 여부와 모델 호출 수를 비교합니다.

2026-11-02 실행 기록의 1회 실행(`RUN_OPTIONAL = True`)에서는 Skill 있음 3문항이 모두 검증을 통과했습니다(`answered` 2건, 질문 2번은 근거가 부족하다고 답한 `insufficient` 1건). Skill 없음 3문항은 모두 `error`, `failed`로 끝났고, 첫 오류는 세 번 모두 `SKILL.md 전체 본문을 read_file로 읽어야 합니다.`였습니다(질문 2번은 `insufficient-evidence/SKILL.md`, 나머지는 `answer/SKILL.md`). 이 셀은 모델 호출 17회, 46.0초가 걸렸습니다. 따라서 Skill 없음의 실패는 답변 품질이 낮아서가 아니라 검증기가 Skill 본문 읽기를 필수로 요구하기 때문에 생긴 것이며, 3문항 1회씩의 관찰이라 일반화할 수는 없습니다. 여러분의 실행에서는 `오류` 칸의 첫 사유가 무엇인지 직접 확인해 보세요.

In [ ]:
# 선택 측정 2: Skill 있음/없음 출력 평가 (RUN_OPTIONAL = True일 때만 실행됩니다)
if not RUN_OPTIONAL:
    print("Skill 있음/없음 비교(질문 3개 x 두 조건, 2026-11-02 실행에서 모델 호출 17회)는 이번에는 돌리지 않았습니다.")
else:
    # [1단계] 빈 폴더, 검색기, 검색 범위(고객 알파, 기준일 2026-09-08)를 준비합니다
    announce("Skill이 있을 때와 없을 때의 RAG 답변을 비교합니다", model_calls=20)
    none_dir = SANDBOX / "no-skills"
    none_dir.mkdir(exist_ok=True)
    retriever = Retriever(load_corpus())
    scope = Scope("알파", date(2026, 9, 8))

    # [2단계] 평가할 SLA 질문 3개와 비교할 두 조건을 정의합니다
    eval_questions = [
        "알파 SLA 표의 서비스 크레딧 비율 중 최댓값은 무엇인가요?",
        "알파 SLA에서 장애 신고는 몇 시간 안에 해야 하나요?",
        "알파 계약의 서비스 크레딧 신청 기한은 언제인가요?",
    ]
    conditions = (("Skill 있음", SKILLS), ("Skill 없음", none_dir))

    # [3단계] 질문마다 두 조건으로 실행하고 상태, 검증, 모델 호출 수, 첫 오류를 출력합니다
    for question in eval_questions:
        for label, skills_dir in conditions:
            rec = ask(
                question,
                retriever=retriever,
                scope=scope,
                model=make_model(),
                max_attempts=1,
                skills_dir=skills_dir,
            )
            trace_counts = rec["trace"]["counts"]
            model_call_count = trace_counts.get("model", 0)
            print(
                f"{label:10s} | 상태={rec['status']:12s} | 검증={rec['validation']:6s} "
                f"| 모델호출={model_call_count} | 오류={rec['errors'][:1]} :: {question[:20]}"
            )

## 확인 질문

1. `SKILL.md`의 메타데이터(`name`, `description`)와 본문 가운데, 모델이 첫 호출의 시스템 프롬프트에서 처음부터 볼 수 있는 것은 무엇인가요?
2. 이 수업에서 정의한 '활성화'의 기준은 무엇이며, `SKILL.md`의 앞 세 줄만 읽은 경우는 왜 활성화로 세지 않나요?
3. `description`을 바꾼 뒤 활성화 결과가 달라졌다면, 질문당 한 번씩만 실행한 관찰로 어디까지 말할 수 있고 무엇은 아직 단정할 수 없나요?
4. 트리거 테스트에서 near-miss(단어는 겹치지만 읽으면 안 되는 질문)가 꼭 필요한 이유를 구체적인 예시 하나를 들어 설명해 보세요.
5. 호출 한도(예산)와 고객·기준일 범위 필터는 왜 Skill 본문에 적지 않고 애플리케이션 코드가 직접 맡아야 하나요?

## 정리

- Skill은 이름과 설명(메타데이터)만 항상 시스템 프롬프트에 실리고, 본문은 모델이 필요하다고 판단해 읽을 때만, 참조 파일은 본문이 가리킬 때만 단계적으로 읽힙니다(점진적 공개).
- 좋은 `description`은 **"무엇을 하는가 + 언제 쓰는가 + 언제 쓰지 않는가(경계)"**를 담습니다. 출력 형식은 본문이나 `output.json`에 두고, 긴 세부 규칙은 참조 파일로 나눕니다.
- Skill은 지침일 뿐 강제가 아닙니다. 모델이 실제로 끝까지 읽었는지는 코드로 확인해야 하며, 한 번 실행해 본 활성화 결과는 통계적 비율이 아니라 1회 관찰입니다.
- 읽어야 하는 질문(positive)뿐 아니라 단어가 겹치지만 읽으면 안 되는 질문(near-miss)까지 시험해야 온전한 트리거 테스트입니다.
- 다음 노트북(`04-Retrieval-Design.ipynb`)에서는 근거를 빠짐없이 정확하게 가져오는 검색 설계를 다루고, 어려운 질문 모음으로 검색 성능을 직접 측정해 봅니다.